# **DATASETS** 
### SIM 2020 / 2021 / 2022 / 2023 / 2024
### SINASC 2020 / 2021 / 2022 / 2023

In [ ]:
import pandas as pd
import numpy as np

sim_dataset = []

sim_dataset.append(pd.read_csv('./SIM/15.csv', sep=";", dtype=str))
sim_dataset.append(pd.read_csv('./SIM/16.csv', sep=";", dtype=str))
sim_dataset.append(pd.read_csv('./SIM/17.csv', sep=";", dtype=str))
sim_dataset.append(pd.read_csv('./SIM/18.csv', sep=";", dtype=str))
sim_dataset.append(pd.read_csv('./SIM/19.csv', sep=";", dtype=str))
sim_dataset.append(pd.read_csv('./SIM/20.csv', sep=";", dtype=str))
sim_dataset.append(pd.read_csv('./SIM/21.csv', sep=";", dtype=str))
sim_dataset.append(pd.read_csv('./SIM/22.csv', sep=";", dtype=str))
sim_dataset.append(pd.read_csv('./SIM/23.csv', sep=";", dtype=str))
sim_dataset.append(pd.read_csv('./SIM/24.csv', sep=";", dtype=str))
sim_dataset.append(pd.read_csv('./SIM/25.csv', sep=";", dtype=str))

sim_dataset = pd.concat(sim_dataset, ignore_index=True)

print(sim_dataset.head())

  ORIGEM TIPOBITO   DTOBITO HORAOBITO NATURAL CODMUNNATU    DTNASC IDADE SEXO  \
0      1        2  18052020      1452     812     120010  07061932   487    1   
1      1        2  20052020      2115     812     120010  16031952   468    2   
2      1        2  21052020      1200     812     120010  17021961   459    2   
3      1        2  21052020      1233     812     120010  10081942   477    2   
4      1        2  22052020      0730     812     120030  13041936   484    2   

  RACACOR  ... FONTES TPRESGINFO TPNIVELINV NUDIASINF DTCADINF MORTEPARTO  \
0       4  ...    NaN        NaN        NaN       NaN      NaN        NaN   
1       4  ...    NaN        NaN        NaN       NaN      NaN        NaN   
2       4  ...    NaN        NaN        NaN       NaN      NaN        NaN   
3       4  ...    NaN        NaN        NaN       NaN      NaN        NaN   
4       4  ...    NaN        NaN        NaN       NaN      NaN        NaN   

  DTCONCASO FONTESINF ALTCAUSA CONTADOR  
0       

In [ ]:
sinasc_dataset = []

sinasc_dataset.append(pd.read_csv('./SINASC/15.csv', sep=";", dtype=str))
sinasc_dataset.append(pd.read_csv('./SINASC/16.csv', sep=";", dtype=str))
sinasc_dataset.append(pd.read_csv('./SINASC/17.csv', sep=";", dtype=str))
sinasc_dataset.append(pd.read_csv('./SINASC/18.csv', sep=";", dtype=str))
sinasc_dataset.append(pd.read_csv('./SINASC/19.csv', sep=";", dtype=str))
sinasc_dataset.append(pd.read_csv('./SINASC/20.csv', sep=";", dtype=str))
sinasc_dataset.append(pd.read_csv('./SINASC/21.csv', sep=";", dtype=str))
sinasc_dataset.append(pd.read_csv('./SINASC/22.csv', sep=";", dtype=str))
sinasc_dataset.append(pd.read_csv('./SINASC/23.csv', sep=";", dtype=str))
sinasc_dataset.append(pd.read_csv('./SINASC/24.csv', sep=";", dtype=str))

sinasc_dataset = pd.concat(sinasc_dataset, ignore_index=True)

print(sinasc_dataset.head())

Os registros do SIM foram filtrados a partir da coluna IDADE, correspondente a idade do falecido, sendo permitidos apenas registros com valores menor a “229”, isto é, menor que 29 dias de idade.

In [ ]:
def filtrar_neonatal(df):
    df = df.copy()

    df["IDADE"] = (
        df["IDADE"]
        .astype(str)
        .str.strip()
    )

    df["IDADE_NUM"] = pd.to_numeric(
        df["IDADE"],
        errors="coerce"
    )

    df_neonatal = df[
        (df["IDADE_NUM"] < 229) &
        (df["IDADE_NUM"] > 0) &
        (df["IDADE_NUM"].notna()) 
    ].copy()

    df_neonatal = df_neonatal.drop(columns=["IDADE_NUM"])

    return df_neonatal

In [ ]:
print(sim_dataset.shape)
sim_neonatal = filtrar_neonatal(sim_dataset)

print(sim_neonatal.shape)

(3389473, 87)
(45292, 87)


No pré-processamento, foram removidos valores da coluna PESO, peso do falecido em gramas, abaixo de 350 e acima de 6.500, de forma similar a [Paixao et al. 2021]

In [ ]:
def filtrar_peso(df, coluna_peso="PESO", peso_min=350, peso_max=6500):
    df = df.copy()

    df[coluna_peso] = df[coluna_peso].astype(str).str.strip()
    df["PESO_NUM"] = pd.to_numeric(df[coluna_peso], errors="coerce")

    n_antes = df.shape[0]

    df = df[
        (df["PESO_NUM"] >= peso_min) &
        (df["PESO_NUM"] <= peso_max)
    ].copy()

    n_depois = df.shape[0]

    print(f"Registros antes: {n_antes}")
    print(f"Registros depois: {n_depois}")
    print(f"Registros removidos: {n_antes - n_depois}")

    df = df.drop(columns=["PESO_NUM"])

    return df

In [ ]:
sim_peso = filtrar_peso(sim_neonatal)

Registros antes: 45292
Registros depois: 41971
Registros removidos: 3321


In [ ]:
sinasc_peso = filtrar_peso(sinasc_dataset)


Registros antes: 5579291
Registros depois: 5574335
Registros removidos: 4956


Colunas categóricas foram criadas para uso junto ao algoritmo preditivo, são elas: catTPROBSON a partir de TPROBSON, código do grupo de Robson, para indicar ocorrência de cesáreas ou partos anteriores, sendo os grupos 1, 2, 3, 4 e 6 considerados como 0 e os demais como 1, similar à variável utilizada em [Ren et al. 2023].

In [ ]:
sinasc_catTprobson = sinasc_peso.copy()
sinasc_catTprobson["TPROBSON_NUM"] = pd.to_numeric(
    sinasc_catTprobson["TPROBSON"].astype(str).str.strip(),
    errors="coerce"
)

grupos = [1, 2, 3, 4, 6]

sinasc_catTprobson["catTPROBSON"] = np.where(
    sinasc_catTprobson["TPROBSON_NUM"].isin(grupos),
    0,
    1
)

sinasc_catTprobson = sinasc_catTprobson.drop(columns=["TPROBSON_NUM"])


In [ ]:
sinasc_catPeso = sinasc_catTprobson.copy()
peso = pd.to_numeric(
    sinasc_catPeso["PESO"].astype(str).str.strip(),
    errors="coerce"
)

sinasc_catPeso["catPeso"] = np.where(
    peso > 2500,
    1,
    0
)

print(sinasc_catTprobson.shape)
print(sinasc_catPeso.shape)

(5574335, 62)
(5574335, 63)


In [ ]:
sinasc_gestac = sinasc_catPeso.copy()

sinasc_gestac["SEMAGESTAC_NUM"] = pd.to_numeric(
    sinasc_gestac["SEMAGESTAC"].astype(str).str.strip(),
    errors="coerce"
)

sinasc_gestac["catSEMAGESTAC"] = np.select(
    [
        sinasc_gestac["SEMAGESTAC_NUM"] < 37,
        (sinasc_gestac["SEMAGESTAC_NUM"] >= 37) & (sinasc_gestac["SEMAGESTAC_NUM"] < 42),
        sinasc_gestac["SEMAGESTAC_NUM"] >= 42
    ],
    [
        "pre_term",
        "term",
        "pos_term"
    ],
    default=None
)

In [ ]:
sim_peso.to_csv("db_sim.csv", index=False)

KeyboardInterrupt: 

In [ ]:
sinasc_gestac.to_csv("db_sinasc.csv", index=False)
